# DQ STG EVENT

Визуализация последнего прогона `dq-stg-event` (логи `DQ_STG_EVENT`): покрытие по ЦОД, микс событий и gate `event.stg_contract`.

Перед запуском: `uv run mobile build-stg-event`, `uv run mobile build-move-event`, затем `uv run mobile dq-stg-event`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_STG_EVENT"
BOUNDARY = ["dataset_presence", "coverage"]

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(80))
    if len(wide) > 80:
        print(f"... всего строк metrics: {len(wide)}")

## Визуализации DQ (только метрики из лога)

Все графики — поля `metrics` из JSON-логов `DQ_STG_EVENT` (без чтения parquet). Прогон режется по `dataset_presence` / `coverage`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    render_stg_event_dq_by_source,
    render_stg_event_dq_overview,
    render_stg_event_dq_quality,
    stg_event_source_coverage_frame,
)

if latest is None:
    print("Пропуск графиков: нет логов DQ.")
else:
    fig = render_stg_event_dq_overview(latest)
    plt.show()
    fig = render_stg_event_dq_quality(latest)
    plt.show()
    fig = render_stg_event_dq_by_source(latest)
    plt.show()
    cov = stg_event_source_coverage_frame(latest)
    if not cov.empty:
        print(cov.to_string(index=False))